In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.special import gammaln
from time import time
from scipy.special import gammaln

In [3]:
# ---------------------------------------------------------------------------
# A_n(z) = E[exp(z R_n)] with R_n ~ chi_n, in log space.  (unchanged)
# ---------------------------------------------------------------------------

def log_A_n_quad(z, n):
    """log A_n(z) by stable numerical integration."""
    z = float(z)

    r_star = (z + np.sqrt(z * z + 4 * (n - 1))) / 2.0
    r_star = max(r_star, 1e-300)

    log_max = (n - 1) * np.log(r_star) - 0.5 * (r_star - z) ** 2

    def integrand(r):
        if r <= 0.0:
            return 0.0
        return np.exp(
            (n - 1) * np.log(r)
            - 0.5 * (r - z) ** 2
            - log_max
        )

    integral, _ = quad(integrand, 0.0, np.inf, limit=200)

    return (
        0.5 * z * z
        - (0.5 * n - 1) * np.log(2)
        - gammaln(0.5 * n)
        + log_max
        + np.log(integral)
    )


def make_log_A_n_table(n, z_max=12.0, num_points=601):
    z_grid = np.linspace(-z_max, z_max, num_points)
    log_A_grid = np.array([log_A_n_quad(z, n) for z in z_grid])
    return z_grid, log_A_grid


def logAn_lookup(z, n, z_grid, log_A_grid):
    z = np.asarray(z, float)
    c_pos = 2.0**(1 - n/2) * np.sqrt(2*np.pi) / np.exp(gammaln(n/2))   # z -> +inf
    c_neg = 2.0**(1 - n/2) * np.exp(gammaln(n)) / np.exp(gammaln(n/2)) # z -> -inf
    zmax = z_grid[-1]
    out = np.interp(z, z_grid, log_A_grid)
    out = np.where(z >  zmax, 0.5*z*z + (n-1)*np.log(np.abs(z)) + np.log(c_pos), out)
    out = np.where(z < -zmax, np.log(c_neg) - n*np.log(np.abs(z)),               out)
    return out

In [4]:
# ---------------------------------------------------------------------------
# Prior log densities (zero-centered, unit scale).
# Used as pi_m(theta) = pi(theta - m).
# ---------------------------------------------------------------------------

def log_pi_gauss(theta):
    return -0.5 * np.log(2.0 * np.pi) - 0.5 * theta * theta


def log_pi_laplace(theta):
    return -np.log(2.0) - np.abs(theta)


def log_pi_t5(theta):
    nu = 5.0
    return (
        gammaln((nu + 1.0) / 2.0)
        - gammaln(nu / 2.0)
        - 0.5 * np.log(nu * np.pi)
        - 0.5 * (nu + 1.0) * np.log1p(theta * theta / nu)
    )


def log_pi_horseshoe(theta):
    """Carvalho-Polson-Scott upper bound, renormalized:
       pi(theta) = (1/(4 pi)) * log(1 + 4/theta^2).
    Polynomial (Cauchy-rate) tails and a soft logarithmic spike at 0."""
    theta_sq = np.maximum(theta * theta, 1e-12)
    return np.log(np.log1p(4.0 / theta_sq) / (4.0 * np.pi))

def log_pi_cauchy(theta):
    """Rouder et al. (2009) default prior on the standardised effect size."""
    return -np.log(np.pi) - np.log1p(theta * theta)

PRIOR_LOG_PDFS = {
    "Cauchy (Rouder)": log_pi_cauchy,
    "Gaussian":   log_pi_gauss,
    "Laplace":    log_pi_laplace,
    "Student-t5": log_pi_t5,
    "Horseshoe":  log_pi_horseshoe,
}

PRIOR_COLORS = {
    "Cauchy (Rouder)": "tab:red",
    "Gaussian":   "tab:blue",
    "Laplace":    "tab:green",
    "Student-t5": "tab:orange",
    "Horseshoe":  "tab:purple",
}

In [5]:
# ---------------------------------------------------------------------------
# Lebesgue prior (closed form, used as the eV baseline).
# ---------------------------------------------------------------------------

def log_M_n_leb(u_n, n):
    return (
        0.5 * np.log(2.0 * np.pi / n)
        - 0.5 * n * np.log1p(-u_n * u_n / n)
    )


def g_leb(c):
    """Closed-form Lebesgue calibration at n = 2."""
    a = np.minimum(np.sqrt(np.pi) / c, 1.0)

    val = (2.0 / np.pi) * (
        np.arcsin(np.sqrt(a))
        + np.sqrt(a * np.maximum(1.0 - a, 0.0))
    )

    return np.where(c <= np.sqrt(np.pi), 1.0, val)

In [6]:
# ---------------------------------------------------------------------------
# General prior: log M_n^{Pi_m}(y) by quadrature in theta, on a u-grid.
# ---------------------------------------------------------------------------

def log_M_n_quad_single(u, m, n, prior_log_pdf, z_grid_n, log_A_grid_n):
    """log integral of  exp(-n theta^2 / 2) * A_n(theta u) * pi(theta - m)  d theta."""

    # Approximate the peak for stable normalization
    candidates = [0.0, float(m), float(m * n / (n + 1.0))]
    log_vals = []
    for theta in candidates:
        log_A = float(logAn_lookup(theta * u, n, z_grid_n, log_A_grid_n))
        log_pi = float(prior_log_pdf(theta - m))
        log_vals.append(-0.5 * n * theta * theta + log_A + log_pi)
    log_max_init = max(log_vals)

    def integrand(theta):
        log_A = float(logAn_lookup(theta * u, n, z_grid_n, log_A_grid_n))
        log_pi = float(prior_log_pdf(theta - m))
        return np.exp(
            -0.5 * n * theta * theta
            + log_A
            + log_pi
            - log_max_init
        )

    theta_max = max(20.0, abs(m) + 12.0)

    interior = sorted({0.0, float(m)})
    interior = [b for b in interior if -theta_max < b < theta_max]

    integral, _ = quad(
        integrand, -theta_max, theta_max,
        limit=200, points=tuple(interior),
    )

    if integral <= 0.0:
        return -np.inf

    return log_max_init + np.log(integral)


# def precompute_log_M_n_on_u_grid(
#     m, n, prior_log_pdf, z_grid_n, log_A_grid_n, n_u=200,
# ):
#     eps_u = 1e-3
#     sqn = np.sqrt(n)
#     u_grid = np.linspace(-sqn + eps_u, sqn - eps_u, n_u)

#     log_M_grid = np.zeros(n_u)
#     for i, u in enumerate(u_grid):
#         log_M_grid[i] = log_M_n_quad_single(
#             u, m, n, prior_log_pdf, z_grid_n, log_A_grid_n,
#         )

#     return u_grid, log_M_grid


#  Densify the u-grid toward the boundary +/- sqrt(n) where M_n is singular,
#    and raise the quad subdivision limit.
def precompute_log_M_n_on_u_grid(m, n, prior_log_pdf, z_grid_n, log_A_grid_n, n_u=200):
    sqn = np.sqrt(n)
    core = np.linspace(-sqn + 1e-3, sqn - 1e-3, n_u)
    edge = sqn - np.geomspace(1e-3, 0.05, 40)
    u_grid = np.unique(np.concatenate([core, edge, -edge]))
    log_M_grid = np.array([
        log_M_n_quad_single(u, m, n, prior_log_pdf, z_grid_n, log_A_grid_n)
        for u in u_grid
    ])
    return u_grid, log_M_grid

In [7]:
# ---------------------------------------------------------------------------
# u_n statistic and eVille cutoff.  (unchanged)
# ---------------------------------------------------------------------------

def u_n_of(Y):
    return Y.sum(axis=-1) / np.sqrt((Y * Y).sum(axis=-1))


def eville_cutoff_from_samples(M2_samples, alpha):
    lo = 1e-12
    hi = max(1.0 / alpha, np.max(M2_samples), M2_samples.mean() / alpha) * 2.0

    for _ in range(80):
        mid = 0.5 * (lo + hi)
        g_mid = np.minimum(M2_samples / mid, 1.0).mean()
        if g_mid <= alpha:
            hi = mid
        else:
            lo = mid

    return hi

In [8]:
# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

rng = np.random.default_rng(seed=20260508)

theta_star = 0.5
alpha = 0.10

n_values = [20, 50, 100, 200]
m_grid = np.linspace(-30.0, 30.0, 30)

B_data = 10000
B_cal = 1000

print("Tabulating log A_n ...")

A_table = {}
for n in [2] + n_values:
    if n not in A_table:
        A_table[n] = make_log_A_n_table(n, z_max=12.0, num_points=601)
        print(f"  n = {n}")

z_grid_n2, log_A_grid_n2 = A_table[2]

Tabulating log A_n ...
  n = 2
  n = 20
  n = 50
  n = 100
  n = 200


In [9]:
# ---------------------------------------------------------------------------
# Calibration: shared H_0 sample, swept over (prior, m).
# ---------------------------------------------------------------------------

print("Drawing H_0 calibration sample ...")

Y1c = rng.standard_normal(B_cal)
Y2c = rng.standard_normal(B_cal)

u2c = (Y1c + Y2c) / np.sqrt(Y1c * Y1c + Y2c * Y2c)


print("\nComputing calibration M_2 samples for each (prior, m) ...")

cal_tables = {}     # (prior_name, m) -> M_2 samples (length B_cal)
t0 = time()

for prior_name, prior_log_pdf in PRIOR_LOG_PDFS.items():
    for m in m_grid:
        u_grid_local, log_M_grid_local = precompute_log_M_n_on_u_grid(
            m=m, n=2,
            prior_log_pdf=prior_log_pdf,
            z_grid_n=z_grid_n2,
            log_A_grid_n=log_A_grid_n2,
            n_u=200,
        )
        log_M_cal = np.interp(u2c, u_grid_local, log_M_grid_local)
        cal_tables[(prior_name, m)] = np.exp(log_M_cal)

    print(f"  {prior_name}: {time() - t0:.1f}s")

Drawing H_0 calibration sample ...

Computing calibration M_2 samples for each (prior, m) ...


/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/1128122159.py:46: RuntimeWarning: divide by zero encountered in log
  out = np.where(z >  zmax, 0.5*z*z + (n-1)*np.log(np.abs(z)) + np.log(c_pos), out)
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/1128122159.py:47: RuntimeWarning: divide by zero encountered in log
  out = np.where(z < -zmax, np.log(c_neg) - n*np.log(np.abs(z)),               out)
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/3177706886.py:32: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  integral, _ = quad(
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/3177706886.py:32: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  I

  Cauchy (Rouder): 166.9s
  Gaussian: 298.4s
  Laplace: 446.9s
  Student-t5: 619.8s


/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/3177706886.py:32: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  integral, _ = quad(


  Horseshoe: 821.7s


In [10]:
# ---------------------------------------------------------------------------
# Power sweep: Ville and extended Ville, all priors, all (n, m).
# ---------------------------------------------------------------------------

print("Power sweep ...")

prior_names = list(PRIOR_LOG_PDFS.keys())

powers_V = {
    (p, n): np.zeros_like(m_grid)
    for p in prior_names for n in n_values
}
powers_eV = {
    (p, n): np.zeros_like(m_grid)
    for p in prior_names for n in n_values
}
leb_baseline = {}

t0 = time()

for n in n_values:
    z_grid_n, log_A_grid_n = A_table[n]

    # one data sample per n, reused across priors and m
    Y_data = rng.standard_normal((B_data, n)) + theta_star
    u_data = u_n_of(Y_data)

    for prior_name, prior_log_pdf in PRIOR_LOG_PDFS.items():
        for j, m in enumerate(m_grid):
            u_grid_local, log_M_grid_local = precompute_log_M_n_on_u_grid(
                m=m, n=n,
                prior_log_pdf=prior_log_pdf,
                z_grid_n=z_grid_n,
                log_A_grid_n=log_A_grid_n,
                n_u=200,
            )
            log_M_data = np.interp(u_data, u_grid_local, log_M_grid_local)
            M_data = np.exp(log_M_data)

            powers_V[(prior_name, n)][j] = (M_data >= 1.0 / alpha).mean()

            c_eV = eville_cutoff_from_samples(
                cal_tables[(prior_name, m)], alpha,
            )
            powers_eV[(prior_name, n)][j] = (M_data >= c_eV).mean()

        print(f"  n = {n}, {prior_name}: {time() - t0:.1f}s")

    # Lebesgue baseline (closed-form, m-independent)
    M_leb = np.exp(log_M_n_leb(u_data, n))
    leb_baseline[n] = (g_leb(M_leb) <= alpha).mean()
    print(f"  n = {n}: Lebesgue eV baseline = {leb_baseline[n]:.3f}")

Power sweep ...


/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/1128122159.py:46: RuntimeWarning: divide by zero encountered in log
  out = np.where(z >  zmax, 0.5*z*z + (n-1)*np.log(np.abs(z)) + np.log(c_pos), out)
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/1128122159.py:47: RuntimeWarning: divide by zero encountered in log
  out = np.where(z < -zmax, np.log(c_neg) - n*np.log(np.abs(z)),               out)
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/3177706886.py:32: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.
  integral, _ = quad(
/var/folders/31/6gx1txds1v51wc6_tdp16jth0000gp/T/ipykernel_15505/3177706886.py:32: IntegrationWarning: The maximum number of subdivisions (200) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  I

  n = 20, Cauchy (Rouder): 130.3s
  n = 20, Gaussian: 272.1s
  n = 20, Laplace: 405.5s
  n = 20, Student-t5: 546.7s
  n = 20, Horseshoe: 688.6s
  n = 20: Lebesgue eV baseline = 0.069
  n = 50, Cauchy (Rouder): 814.8s
  n = 50, Gaussian: 960.2s
  n = 50, Laplace: 1087.7s
  n = 50, Student-t5: 1227.8s
  n = 50, Horseshoe: 1365.2s
  n = 50: Lebesgue eV baseline = 0.387
  n = 100, Cauchy (Rouder): 1493.7s
  n = 100, Gaussian: 1621.1s
  n = 100, Laplace: 1748.2s
  n = 100, Student-t5: 1886.2s


: 

In [1]:
# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(
    2, len(n_values),
    figsize=(4.5 * len(n_values), 6.5),
    sharex=True, sharey=True,
)

for k, n in enumerate(n_values):
    # Top row: Ville
    ax = axes[0, k]

    for prior_name in prior_names:
        ax.plot(
            m_grid, powers_V[(prior_name, n)],
            "-o", markersize=3,
            color=PRIOR_COLORS[prior_name],
            label=prior_name,
        )

    ax.axvline(theta_star, color="gray", linestyle="--", linewidth=1)
    ax.axhline(alpha, color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"Ville,  $n = {n}$")
    ax.set_ylim(-0.02, 1.02)

    if k == 0:
        ax.set_ylabel("Empirical power")
        ax.legend(loc="upper left", fontsize=8, frameon=True)

    # Bottom row: extended Ville
    ax = axes[1, k]

    for prior_name in prior_names:
        ax.plot(
            m_grid, powers_eV[(prior_name, n)],
            "-o", markersize=3,
            color=PRIOR_COLORS[prior_name],
            linewidth=(2.0 if prior_name == "Cauchy (Rouder)" else 1.0),
            label=prior_name,
        )

    ax.axhline(
        leb_baseline[n],
        color="black", linestyle="-.", linewidth=1.2,
        label="Lebesgue (improper)",
    )

    ax.axvline(theta_star, color="gray", linestyle="--", linewidth=1)
    ax.axhline(alpha, color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"Extended Ville,  $n = {n}$")
    ax.set_xlabel(r"Prior mean  $m$")
    ax.set_ylim(-0.02, 1.02)

    if k == 0:
        ax.set_ylabel("Empirical power")
        ax.legend(loc="upper left", fontsize=8, frameon=True)

fig.suptitle(
    rf"Power vs prior mean ($\theta^\star={theta_star}$, $\alpha={alpha}$)",
    y=1.02,
)
fig.tight_layout()
fig.savefig("power_vs_m_priors_v4.pdf", bbox_inches="tight")
plt.show()

NameError: name 'plt' is not defined